## Setup and Environment Check

In [ ]:
# Environment verification
import os
import sys
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, '../../src')

# Verify we're in research environment
if os.getenv('GALACTUS_ENV') == 'production':
    raise RuntimeError("Cannot run research experiments in production!")

print("✓ Environment check passed")
print("✓ Running in research mode")
print(f"✓ Python path: {sys.executable}")
print(f"✓ Working directory: {Path.cwd()}")

## Import Research Tools and Signal

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, List
from dataclasses import asdict

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

# Import Galactus research tools
from features.isolation import mark_experimental
from features.oi_decay_research import (
    compute_oi_decay_pressure,
    validate_oi_decay_signal,
    generate_synthetic_oi_data,
    analyze_oi_decay_patterns,
    OIDecayConfig,
    OIDecayResult
)

# Import validation framework
from validation.walk_forward import ValidationWindow, WindowType

print("✓ All imports successful")

## 1. Signal Logic Verification

Test the core signal computation with known inputs and expected outputs.

In [ ]:
# Test basic functionality
print("=== OI Decay Signal Logic Verification ===\n")

# Test case 1: Strong decay (unwinding)
current_oi_decay = {18000: 500, 18500: 800, 19000: 600}  # Reduced OI
previous_oi_decay = {18000: 1000, 18500: 1500, 19000: 1200}  # Original OI

result_decay = compute_oi_decay_pressure(current_oi_decay, previous_oi_decay, 1.0)
print(f"Strong Decay Test:")
print(f"  Pressure: {result_decay.pressure:.3f} (expected: negative)")
print(f"  Total Decay: {result_decay.total_decay}")
print(f"  Confidence: {result_decay.confidence:.3f}")
print()

# Test case 2: Strong building
current_oi_build = {18000: 1500, 18500: 2000, 19000: 1800}  # Increased OI
previous_oi_build = {18000: 1000, 18500: 1500, 19000: 1200}  # Original OI

result_build = compute_oi_decay_pressure(current_oi_build, previous_oi_build, 1.0)
print(f"Strong Building Test:")
print(f"  Pressure: {result_build.pressure:.3f} (expected: positive)")
print(f"  Total Build: {result_build.total_build}")
print(f"  Confidence: {result_build.confidence:.3f}")
print()

# Test case 3: No change
result_no_change = compute_oi_decay_pressure(previous_oi_decay, previous_oi_decay, 1.0)
print(f"No Change Test:")
print(f"  Pressure: {result_no_change.pressure:.3f} (expected: 0.0)")
print(f"  Confidence: {result_no_change.confidence:.3f}")
print()

## 2. Synthetic Data Testing

Generate synthetic OI data to test signal behavior under controlled conditions.

In [ ]:
# Generate synthetic test data
print("=== Synthetic Data Testing ===\n")

# Base OI levels (typical NIFTY strikes)
base_oi = {
    17000: 500,   # OTM Put
    17500: 1200,  # OTM Put  
    18000: 2500,  # ATM-ish
    18500: 1800,  # OTM Call
    19000: 800,   # OTM Call
    19500: 300    # Deep OTM Call
}

# Test different decay rates
decay_rates = [-0.8, -0.4, 0.0, 0.4, 0.8]  # Strong decay to strong building
time_delta = 1.0  # 1 hour

synthetic_results = []

for decay_rate in decay_rates:
    # Generate synthetic current OI
    current_oi = generate_synthetic_oi_data(base_oi, decay_rate, time_delta, noise_factor=0.05)
    
    # Compute signal
    result = compute_oi_decay_pressure(current_oi, base_oi, time_delta)
    
    synthetic_results.append({
        'decay_rate': decay_rate,
        'pressure': result.pressure,
        'confidence': result.confidence,
        'total_decay': result.total_decay,
        'total_build': result.total_build
    })
    
    print(f"Decay Rate {decay_rate:+.1f}: Pressure = {result.pressure:+.3f}, Confidence = {result.confidence:.3f}")

print("\n✓ Synthetic data testing completed")

In [ ]:
# Visualize synthetic results
df_synthetic = pd.DataFrame(synthetic_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pressure vs Decay Rate
ax1.plot(df_synthetic['decay_rate'], df_synthetic['pressure'], 'bo-', linewidth=2, markersize=8)
ax1.plot([-1, 1], [-1, 1], 'r--', alpha=0.5, label='Ideal (slope=1)')
ax1.set_xlabel('Input Decay Rate')
ax1.set_ylabel('Computed Pressure')
ax1.set_title('Signal Response to Decay Rate')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Confidence levels
ax2.bar(range(len(df_synthetic)), df_synthetic['confidence'], 
        tick_label=[f'{x:+.1f}' for x in df_synthetic['decay_rate']])
ax2.set_xlabel('Decay Rate')
ax2.set_ylabel('Confidence')
ax2.set_title('Signal Confidence by Decay Rate')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Statistical Properties Analysis

Analyze the statistical properties of the signal under various conditions.

In [ ]:
# Statistical analysis of signal properties
print("=== Statistical Properties Analysis ===\n")

# Generate large sample of random OI changes
np.random.seed(42)  # For reproducibility
n_samples = 1000

statistical_results = []
config = OIDecayConfig()

for i in range(n_samples):
    # Generate random OI data
    decay_rate = np.random.uniform(-1, 1)
    time_delta = np.random.uniform(0.5, 24.0)  # 30 min to 24 hours
    
    current_oi = generate_synthetic_oi_data(base_oi, decay_rate, time_delta, noise_factor=0.1)
    
    result = compute_oi_decay_pressure(current_oi, base_oi, time_delta, config)
    
    statistical_results.append({
        'input_decay': decay_rate,
        'time_delta': time_delta,
        'pressure': result.pressure,
        'confidence': result.confidence,
        'strike_count': result.strike_count
    })

df_stats = pd.DataFrame(statistical_results)

# Statistical summary
print("Signal Distribution Statistics:")
print(df_stats[['pressure', 'confidence']].describe())
print(f"\nSignal Range: {df_stats['pressure'].min():.3f} to {df_stats['pressure'].max():.3f}")
print(f"Mean Absolute Pressure: {abs(df_stats['pressure']).mean():.3f}")
print(f"Confidence Range: {df_stats['confidence'].min():.3f} to {df_stats['confidence'].max():.3f}")

# Correlation analysis
correlation = df_stats['pressure'].corr(df_stats['input_decay'])
print(f"\nCorrelation with Input Decay Rate: {correlation:.3f}")

# Test signal responsiveness
strong_decay = df_stats[df_stats['input_decay'] < -0.5]
strong_build = df_stats[df_stats['input_decay'] > 0.5]

print(f"\nStrong Decay Cases (input < -0.5): mean pressure = {strong_decay['pressure'].mean():+.3f}")
print(f"Strong Build Cases (input > +0.5): mean pressure = {strong_build['pressure'].mean():+.3f}")

In [ ]:
# Visualize statistical properties
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Pressure distribution
axes[0,0].hist(df_stats['pressure'], bins=50, alpha=0.7, edgecolor='black')
axes[0,0].set_xlabel('Pressure')
axes[0,0].set_ylabel('Frequency')
axes[0,0].set_title('Pressure Distribution')
axes[0,0].grid(True, alpha=0.3)

# Confidence distribution
axes[0,1].hist(df_stats['confidence'], bins=50, alpha=0.7, edgecolor='black', color='green')
axes[0,1].set_xlabel('Confidence')
axes[0,1].set_ylabel('Frequency')
axes[0,1].set_title('Confidence Distribution')
axes[0,1].grid(True, alpha=0.3)

# Pressure vs Input Decay
axes[1,0].scatter(df_stats['input_decay'], df_stats['pressure'], alpha=0.6, s=10)
axes[1,0].plot([-1, 1], [-1, 1], 'r--', linewidth=2, label='Ideal Response')
axes[1,0].set_xlabel('Input Decay Rate')
axes[1,0].set_ylabel('Computed Pressure')
axes[1,0].set_title('Signal Response Correlation')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Time delta effect
axes[1,1].scatter(df_stats['time_delta'], df_stats['pressure'], alpha=0.6, s=10, c=df_stats['input_decay'], cmap='coolwarm')
axes[1,1].set_xlabel('Time Delta (hours)')
axes[1,1].set_ylabel('Pressure')
axes[1,1].set_title('Time Delta Effect on Pressure')
axes[1,1].colorbar(label='Input Decay Rate')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Failure Mode Analysis

Test the signal under conditions known to cause failures.

In [ ]:
# Failure mode testing
print("=== Failure Mode Analysis ===\n")

failure_tests = [
    {
        'name': 'Insufficient Strike Data',
        'current': {18000: 100},
        'previous': {18000: 200},
        'time_delta': 1.0,
        'expected_failure': 'insufficient_data'
    },
    {
        'name': 'No OI Change',
        'current': base_oi.copy(),
        'previous': base_oi.copy(),
        'time_delta': 1.0,
        'expected_failure': 'no_change'
    },
    {
        'name': 'Invalid Time Delta',
        'current': base_oi.copy(),
        'previous': {k: v+100 for k, v in base_oi.items()},
        'time_delta': -1.0,
        'expected_failure': 'ValueError'
    },
    {
        'name': 'Extreme Decay (Expiry-like)',
        'current': {k: max(0, v//10) for k, v in base_oi.items()},  # 90% decay
        'previous': base_oi.copy(),
        'time_delta': 1.0,
        'expected_failure': None  # Should work but with extreme values
    }
]

for test in failure_tests:
    print(f"Testing: {test['name']}")
    
    try:
        result = compute_oi_decay_pressure(
            test['current'], 
            test['previous'], 
            test['time_delta']
        )
        
        validation = validate_oi_decay_signal(result)
        
        print(f"  Result: pressure={result.pressure:.3f}, confidence={result.confidence:.3f}")
        print(f"  Validation: {'✓' if validation['is_valid'] else '✗'} {validation['warnings']}")
        
        if test['expected_failure']:
            if test['expected_failure'] in str(result.metadata):
                print(f"  ✓ Expected failure detected: {test['expected_failure']}")
            else:
                print(f"  ⚠ Unexpected success, expected: {test['expected_failure']}")
        
    except Exception as e:
        if test['expected_failure'] and test['expected_failure'] in str(e):
            print(f"  ✓ Expected exception: {type(e).__name__}: {e}")
        else:
            print(f"  ✗ Unexpected exception: {type(e).__name__}: {e}")
    
    print()

## 5. Walk-Forward Validation Setup

Set up the framework for walk-forward validation (to be run with real data).

In [ ]:
# Walk-forward validation setup
print("=== Walk-Forward Validation Setup ===\n")

# Define validation windows (example structure)
validation_windows = [
    ValidationWindow(
        window_type=WindowType.TRAINING,
        start_time=datetime(2024, 1, 1),
        end_time=datetime(2024, 6, 30),
        regime_labels=["normal", "elevated_volatility"],
        justification="Initial training period for signal development"
    ),
    ValidationWindow(
        window_type=WindowType.VALIDATION,
        start_time=datetime(2024, 7, 1),
        end_time=datetime(2024, 12, 31),
        regime_labels=["normal", "expiry", "earnings"],
    )
]

print("Validation Windows Defined:")
for i, window in enumerate(validation_windows, 1):
    print(f"{i}. {window.window_type.value.title()} Window:")
    print(f"   Period: {window.start_time.date()} to {window.end_time.date()}")
    print(f"   Duration: {window.duration_days} days")
    print(f"   Regimes: {window.regime_labels}")
    if hasattr(window, 'justification') and window.justification:
        print(f"   Justification: {window.justification}")
    print()

# Define success criteria for validation
success_criteria = {
    'statistical_significance': {
        'p_value_threshold': 0.05,
        'minimum_sample_size': 100,
        'required_regimes': ['normal', 'elevated_volatility']
    },
    'signal_quality': {
        'min_correlation_with_events': 0.3,
        'max_false_positive_rate': 0.2,
        'min_confidence_threshold': 0.4
    },
    'robustness': {
        'regime_stability_required': True,
        'data_quality_sensitivity_test': True,
        'walk_forward_consistency': True
    }
}

print("Success Criteria Defined:")
for category, criteria in success_criteria.items():
    print(f"{category.replace('_', ' ').title()}:")
    for metric, value in criteria.items():
        print(f"  - {metric.replace('_', ' ')}: {value}")
    print()

## 6. Experiment Summary and Next Steps

Document the experiment results and outline next steps.

In [ ]:
# Experiment summary
print("=== Experiment EXP-2025-12-001 Summary ===\n")

experiment_summary = {
    'experiment_id': 'EXP-2025-12-001',
    'hypothesis': 'Derivatives OI decay rate indicates forced position unwinding due to capital constraints',
    'signal_implementation': 'oi_decay_research.py',
    'notebook_location': 'research/python/notebooks/exploratory/oi_decay_experiment.ipynb',
    'completion_date': datetime.now().date(),
    'status': 'research_phase_complete',
    'findings': {
        'signal_logic': '✓ Implemented and tested',
        'synthetic_testing': '✓ Completed with good correlation',
        'statistical_properties': '✓ Analyzed distribution and responsiveness',
        'failure_modes': '✓ Identified and tested',
        'validation_framework': '✓ Set up for walk-forward testing'
    },
    'key_metrics': {
        'correlation_with_input': correlation,
        'signal_range': [-1.0, 1.0],
        'typical_confidence': df_stats['confidence'].mean(),
        'failure_cases_handled': len([t for t in failure_tests if t['expected_failure']])
    },
    'next_steps': [
        'Run walk-forward validation with real market data',
        'Test signal during known market events (expiry, earnings)',
        'Compare with alternative OI-based signals',
        'Prepare promotion checklist if validation succeeds'
    ],
    'promotion_readiness': 'pending_validation',
    'risks_identified': [
        'Signal may be noisy during low liquidity periods',
        'Expiry effects need careful filtering',
        'Time delta measurement accuracy critical',
        'Strike selection methodology affects results'
    ]
}

print("Experiment Summary:")
print(f"ID: {experiment_summary['experiment_id']}")
print(f"Hypothesis: {experiment_summary['hypothesis']}")
print(f"Status: {experiment_summary['status']}")
print(f"Completion: {experiment_summary['completion_date']}")
print("\nKey Findings:")
for finding, status in experiment_summary['findings'].items():
    print(f"  {finding.replace('_', ' ').title()}: {status}")
print("\nKey Metrics:")
for metric, value in experiment_summary['key_metrics'].items():
    print(f"  {metric.replace('_', ' ').title()}: {value}")
print("\nNext Steps:")
for i, step in enumerate(experiment_summary['next_steps'], 1):
    print(f"  {i}. {step}")
print("\nRisks Identified:")
for risk in experiment_summary['risks_identified']:
    print(f"  • {risk}")

print(f"\n🎯 Promotion Readiness: {experiment_summary['promotion_readiness'].replace('_', ' ').title()}")
print("\n📝 This experiment provides the foundation for OI decay signal validation.")
print("Proceed to real data validation before considering promotion to production.")